# Lesson 5: Layers and regions

**You will learn:** how to record several layers, decode information from them on held-out images, and compare a trained network with the same network untrained.
**Requires:** CPU only, about a minute.

**The question:** can animals be told from vehicles from each layer, and how much of that comes from training? An untrained network with the same architecture is the null: images differ in colour and texture, so even random features can separate some categories.

## Setup

Imports and settings.

In [1]:
# Keep the output short: no progress bars.
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

ASSETS = Path('../assets/lessons')        # weights and images committed with the lessons

# SqueezeNet 1.1 trained on ImageNet (5 MB). These images come from another dataset, CIFAR-10.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()                                # inference only: no dropout, no training

# The network expects 224 x 224 images, normalized the way it was trained.
TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

A `BrainScoreModel` is assembled from parts. The **extractor** prepares images and records layers; the **region map** says which layer stands for which brain area (a hypothesis, not a fact).

In [2]:
from brainscore import BrainScoreModel
from brainscore_vision.model_helpers.activations.pytorch import PytorchWrapper

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': PytorchWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'})

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(subject.out_channels)                          # what this subject can produce

['neural:IT', 'neural:V1', 'neural:V4']

The null subject is the same architecture with random weights, built the same way.

In [3]:
torch.manual_seed(0)                                           # fixed seed: the same random weights every run
untrained = torchvision.models.squeezenet1_1().eval()
null_subject = squeezenet_subject(untrained, 'squeezenet1_1-untrained')

## The experiment

### Stimuli

In [4]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images, 12 of each CIFAR-10 category, each labelled animal or vehicle.
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

### Task

Plain viewing: each image is shown once and three regions, V1, V4 and IT, are recorded. There is no answer to give.

## Run

One session per network; all three regions come from one pass.

In [ ]:
from brainscore_core.streaming_helpers import stimulus_session

regions = ['V1', 'V4', 'IT']
subjects = {'trained': subject, 'untrained': null_subject}

recordings = {}                                                # (network, region) -> assembly
for name, s in subjects.items():
    session = stimulus_session(stimuli, [f'neural:{r}' for r in regions])
    s.participate(session)
    for region in regions:
        recordings[name, region] = session.collect(f'neural:{region}')

## Read results back

One assembly per network and region: one row per image, one column per unit.

In [6]:
pd.DataFrame([{'network': name, 'region': region, 'shape': assembly.shape}
              for (name, region), assembly in recordings.items()])

,network,region,shape
0,trained,V1,"(120, 193600)"
1,trained,V4,"(120, 93312)"
2,trained,IT,"(120, 86528)"
3,untrained,V1,"(120, 193600)"
4,untrained,V4,"(120, 93312)"
5,untrained,IT,"(120, 86528)"


## Analysis

Five-fold cross-validation with a regularized logistic classifier: every image is scored by a classifier that never saw it.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

folds = StratifiedKFold(5, shuffle=True, random_state=0)       # same five splits for every layer
decoder = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000))

def decoding_accuracy(X, y):
    """Mean held-out accuracy of the decoder over the five folds."""
    return cross_val_score(decoder, X, y, cv=folds).mean()

## Results

In [8]:
y = np.asarray(stimuli['label'])
rows = []
for (name, region), assembly in recordings.items():
    X = np.asarray(assembly.values)
    accuracy = decoding_accuracy(X, y)
    rows.append({'network': name, 'region': region, 'units': X.shape[1], 'accuracy': accuracy})

table = pd.DataFrame(rows).pivot(index='region', columns='network', values='accuracy').loc[regions]
table['trained minus untrained'] = table['trained'] - table['untrained']
table.round(3)

network,trained,untrained,trained minus untrained
region,,,
V1,0.808,0.717,0.092
V4,0.875,0.800,0.075
IT,0.867,0.808,0.058


Chance is 0.6 (72 animals, 48 vehicles). Read the last column: the part of each layer's accuracy that training added. Whether it grows from V1 to IT is the answer to the question, and it need not.

## Summary

- One session records several regions in one pass.
- Decode on held-out images, and compare with an untrained network of the same architecture.
- The region map is a hypothesis; this kind of comparison is how you test it.

**Next:** [Lesson 6](06_behavior.ipynb) turns a layer into answers.